# Logic puzzles in Prolog

Four classic puzzles, each one short program run by SCRIP's Prolog. Click **Run > Run All Cells**, or run a cell at a time with Shift+Enter, and read the answer under each cell. Hover a name for its definition, press Tab to complete one, and use the debugger to step through a puzzle.

## The zebra puzzle

Five houses, five owners, fifteen clues. Who owns the zebra? The program states the clues and lets Prolog's search find the one arrangement that fits them all.

In [ ]:
% attribute store is 'Name - Value' pairs with unique names
attrs( H, [N-V | R]) :- !, memberchk( N-X, H), X = V, (R = [], ! ; attrs( H, R)).
attrs( HS, AttrsL) :- maplist( attrs, HS, AttrsL).

in(    HS, Attrs) :- in( member, HS, Attrs).
in( G, HS, Attrs) :- call( G, A, HS), attrs( A, Attrs).

left_of( [A,B], HS) :- append( _, [A,B | _], HS).
next_to( [A,B], HS) :- left_of( [A,B], HS) ; left_of( [B,A], HS).

zebra( Owner, Houses):-
    Houses = [A,_,C,_,_],                                               % 1
    maplist( in(Houses), [ [ nation-englishman,   color-red          ]  % 2
                         , [ nation-swede,        owns -dog          ]  % 3
                         , [ nation-dane,         drink-tea          ]  % 4
                         , [ drink -coffee,       color-green        ]  % 6
                         , [ smoke -'Pall Mall',  owns -birds        ]  % 7
                         , [ color -yellow,       smoke-'Dunhill'    ]  % 8
                         , [ drink -beer,         smoke-'Blue Master']  % 13
                         , [ nation-german,       smoke-'Prince'     ]  % 14
                         ] ),
    in( left_of, Houses,   [[color -green    ],  [color -white    ]]),  % 5
    in( left_of,  [C,A],   [[drink -milk     ],  [nation-norwegian]]),  % 9, 10
    maplist( in( next_to, Houses),
                         [ [[smoke -'Blend'  ],  [owns -cats      ]]    % 11
                         , [[owns  -horse    ],  [smoke-'Dunhill' ]]    % 12
                         , [[nation-norwegian],  [color-blue      ]]    % 15
                         , [[drink -water    ],  [smoke-'Blend'   ]]    % 16
                         ] ),
    in( Houses, [owns-zebra, nation-Owner]).

% Run it and print the answer:
:- initialization((zebra(Owner, Houses), maplist(length, Houses, _), forall(member(H, Houses), format("~w~n", [H])), format("~w owns the zebra~n", [Owner]))).

## SEND + MORE = MONEY

Each letter stands for a different digit, and the sum must be right. This is the plain constraint version, with the CLP(FD) library: the constraints narrow every letter's digits first, then `labeling` picks the digits.

In [ ]:
:- use_module(library(clpfd)).

send_more_money(Digits) :-
        Digits = [S,E,N,D,M,O,R,Y],
        Digits ins 0..9,
        all_distinct(Digits),
        S #> 0,
        M #> 0,
               1000*S + 100*E + 10*N + D
        +      1000*M + 100*O + 10*R + E
        #= 10000*M + 1000*O + 100*N + 10*E + Y.

:- initialization((send_more_money(Ds), labeling([ff], Ds), Ds = [S,E,N,D,M,O,R,Y],
                   format("SEND = ~w~w~w~w~nMORE = ~w~w~w~w~nMONEY = ~w~w~w~w~w~n", [S,E,N,D,M,O,R,E,M,O,N,E,Y]))).

## Sudoku

A hard puzzle with only a handful of digits given. The rules are three constraints, every row, every column and every 3 by 3 block holds the digits 1 to 9 once, and the library does the rest.

In [ ]:
:- use_module(library(clpfd)).

sudoku(Rows) :-
        length(Rows, 9), maplist(length_(9), Rows),
        append(Rows, Vs), Vs ins 1..9,
        maplist(all_distinct, Rows),
        transpose(Rows, Columns), maplist(all_distinct, Columns),
        Rows = [A,B,C,D,E,F,G,H,I],
        blocks(A, B, C), blocks(D, E, F), blocks(G, H, I).

length_(L, Ls) :- length(Ls, L).

blocks([], [], []).
blocks([A,B,C|Bs1], [D,E,F|Bs2], [G,H,I|Bs3]) :-
        all_distinct([A,B,C,D,E,F,G,H,I]),
        blocks(Bs1, Bs2, Bs3).

problem(1, [[_,_,_,_,_,_,_,_,_],
            [_,_,_,_,_,3,_,8,5],
            [_,_,1,_,2,_,_,_,_],
            [_,_,_,5,_,7,_,_,_],
            [_,_,4,_,_,_,1,_,_],
            [_,9,_,_,_,_,_,_,_],
            [5,_,_,_,_,_,_,7,3],
            [_,_,2,_,1,_,_,_,_],
            [_,_,_,_,4,_,_,_,9]]).

% Run it and print the answer:
:- initialization((problem(1, Rows), sudoku(Rows), maplist(label, Rows), forall(member(R, Rows), format("~w~n", [R])))).

## The towers of Hanoi

Move three disks from the left pole to the centre, one at a time, never a larger disk on a smaller one. The classic recursion states it in three lines.

In [ ]:
hanoi(N) :- move(N,left,center,right).

move(0,_,_,_) :- !.
move(N,A,B,C) :-
    M is N-1,
    move(M,A,C,B),
    inform(A,B),
    move(M,C,B,A).

inform(X,Y) :- write([move,a,disk,from,the,X,pole,to,Y,pole]), nl.

% Run it and print the answer:
:- initialization(hanoi(3)).